# Tutorial: Flattening Search Trees for GPUs

In this tutorial, we explore **Tensorization**—the process of converting complex, hierarchical `SearchTree` objects into flat, batch-friendly PyTorch Tensors.

### Why do we do this?
1. **GPU Efficiency:** GPUs hate following pointers. They love big, contiguous matrices.
2. **Batching Irregularity:** Every search tree has a different shape. We need a way to process a batch of 100 trees with different depths as if they were one giant graph.
3. **The GNN Sweep:** To run the topological sweep (leaves-to-root), we need to quickly query nodes by their depth.

In [ ]:
import os
import sys
from pathlib import Path

# Setup PATH for Graphviz
os.environ["PATH"] += os.pathsep + "/opt/homebrew/bin"

# Setup path to import core modules
LMCOS_DIR = Path(os.getcwd()).parent
if str(LMCOS_DIR) not in sys.path:
    sys.path.insert(0, str(LMCOS_DIR))

import torch
from tensorizer import TreeTensorizer
from schema import tree_encoder_feature_schema
from helper_tensorization import build_demo_trees, summarize_batch_structure, plot_tree, visualize_batch_layout, visualize_gnn_sweep_step

## Step 1: Create the Source Trees

We'll start with two trees of different shapes:
- **Tree A:** A broad, shallow fork (Root with 2 children).
- **Tree B:** A narrow, deep line (Root -> Child -> Grandchild).

In [ ]:
trees = build_demo_trees()
for i, t in enumerate(trees):
    display(plot_tree(t, f"Tree {chr(65+i)}"))

## Step 2: The "Flattening" (Tensorization)

We use the `TreeTensorizer` to pack these two trees into a single `TreeBatch`.

Under the hood, this:
1. Concatenates all node features into one large matrix.
2. Adjusts parent/child indices so Tree 2 doesn't accidentally point into Tree 1.
3. Pre-calculates `depth` to enable the topological GNN sweep.

In [ ]:
schema = tree_encoder_feature_schema()
tensorizer = TreeTensorizer(schema)

batch = tensorizer.tensorize_forest(trees)

summarize_batch_structure(batch)
display(visualize_batch_layout(batch))

## Step 3: Visualizing the GNN Execution Plan

This is the **Topological Sweep**. Instead of traversing linked lists, the GNN simply filters the batch by `depth`.

### The Upward Pass (Orange)
Data flows from leaves to roots. Notice how Tree A finishes before Tree B because it is shallower.

In [ ]:
max_depth = int(batch.depth.max())
for d in range(max_depth, -1, -1):
    display(visualize_gnn_sweep_step(batch, d, direction='up'))

### The Downward Pass (Green)
Global context from the root flows down to the leaves.

In [ ]:
for d in range(max_depth + 1):
    display(visualize_gnn_sweep_step(batch, d, direction='down'))

### Conclusion
By transforming hierarchical trees into these "Structural Tensors", we've prepared them for mass processing.

This architecture allows us to run search-based reasoning as a single, massive GPU kernel sweep, regardless of how irregular the individual search trees are.